In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torchvision
import timm
import torchxrayvision as xrv
import copy
import os

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


## Loading the Data

In [4]:
# Data Transforms
input_size = 224
batch_size = 64

# Subset Training?
subset = True

mean = -5.6936
std = 534.7731

transforms = torchvision.transforms.Compose([
    xrv.datasets.XRayCenterCrop(), 
    xrv.datasets.XRayResizer(input_size), 
    xrv.datasets.transforms.ToTensor(), 
    xrv.datasets.transforms.Normalize(mean, std)])

In [5]:
model_save_path = 'E:/Models/'
path            = 'E:/Datasets/rsna-pneumonia-detection-challenge/stage_2_train_images'

dataset = xrv.datasets.RSNA_Pneumonia_Dataset(imgpath=path, extension='.dcm', transform=transforms)

if subset:
    indices       = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(42))[:3600].tolist()
    train_dataset = Subset(dataset, indices[:3000])
    val_dataset   = Subset(dataset, indices[3000:3300])
    test_dataset  = Subset(dataset, indices[3300:])
else:
    splits = (11511, 1500, 1500)
    train_dataset, val_dataset, test_dataset = random_split(dataset, splits, torch.Generator().manual_seed(42))

In [6]:
train_loader = DataLoader(train_dataset, batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size, shuffle=False)

dataloaders  = {'train':train_loader, 'val':val_loader, 'test':test_loader}

In [7]:
# sample = iter(train_loader).next()
# plt.imshow(sample['img'][0].permute(2, 0, 1).detach().numpy(), cmap="Greys_r")

## Loading the Model

In [17]:
model_name = 'swin_base_patch4_window7_224'
model = timm.create_model(model_name, pretrained=True, in_chans=1, num_classes=2).to(device)

In [18]:
pytorch_total_params = sum(p.numel() for p in model.parameters())
print(pytorch_total_params)

86741178


In [8]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

In [9]:
def train(model, dataloaders, criterion, optimizer, scheduler=None, num_epochs=15, device='cuda'):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.
    best_loss = 10000

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ('train', 'val'):
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.
            running_corrects = 0
            num_inputs = 0.
            for data in dataloaders[phase]:
                inputs = data['img'].permute(0, 2, 3, 1).to(device)
                labels = data['lab'][:, 0].to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels.long())
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                num_inputs += inputs.size(0)
            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / num_inputs
            epoch_acc  = running_corrects.double() / num_inputs

            print(f'{phase} Loss: {epoch_loss} | Acc: {epoch_acc}')

            # deep copy the model
            if phase == 'val' and epoch_loss < best_loss:
                best_loss = epoch_loss
                best_model_wts = copy.deepcopy(model.state_dict())
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
        print()

    print(f'Best val Acc: {best_acc:4f} | loss: {best_loss:4f}')

    # load best model weights
    model.load_state_dict(best_model_wts)
    return model

In [10]:
def test(model, dataloaders, device='cuda'):
    with torch.no_grad():
        n_correct = 0
        n_samples = 0

        for data in dataloaders['test']:

            features = data['img'].permute(0, 2, 3, 1).to(device)
            labels   = data['lab'][:, 0].to(device)

            outputs = model(features)
            _, predictions = torch.max(outputs, 1)
            n_samples += labels.shape[0]
            n_correct += (predictions == labels).sum().item()

        return 100 * n_correct / n_samples

In [11]:
model = train(model, dataloaders, criterion, optimizer, scheduler)

Epoch 0/14
----------
train Loss: 0.42198885401089986 | Acc: 0.8896666666666666
val Loss: 0.39707292397816973 | Acc: 0.8700000000000001

Epoch 1/14
----------
train Loss: 0.32647482593854266 | Acc: 0.9003333333333333
val Loss: 0.38157827854156495 | Acc: 0.8700000000000001

Epoch 2/14
----------
train Loss: 0.3214219672679901 | Acc: 0.9003333333333333
val Loss: 0.3927832319339116 | Acc: 0.8700000000000001

Epoch 3/14
----------
train Loss: 0.3235186026096344 | Acc: 0.9003333333333333
val Loss: 0.38530311981836957 | Acc: 0.8700000000000001

Epoch 4/14
----------
train Loss: 0.31981725748380024 | Acc: 0.9003333333333333
val Loss: 0.37506219387054446 | Acc: 0.8700000000000001

Epoch 5/14
----------
train Loss: 0.31586265873909 | Acc: 0.9003333333333333
val Loss: 0.372301540573438 | Acc: 0.8700000000000001

Epoch 6/14
----------
train Loss: 0.3155723742643992 | Acc: 0.9003333333333333
val Loss: 0.3683523162206014 | Acc: 0.8700000000000001

Epoch 7/14
----------
train Loss: 0.311813532352447

In [12]:
torch.save(model, f'{model_save_path}{model_name}3k.pth')

In [13]:
test(model, dataloaders)

90.0

Parameter counts:
1. vit_tiny - 5,426,498
2. vit_base - 85,406,978
3. swin     - 86,741,178
4. resnet18 - 11,171,266
5. resnet50 - 23,505,858

* 1.5k
1. Model: vit_base_patch16_224 | Accuracy: 91.0
2. Model: vit_tiny_patch16_224 | Accuracy: 91.4
3. Model: resnet18 | Accuracy: 91.8
4. Model: resnet50 | Accuracy: 91.0
5. Model: swin_base_patch4_window7_224 | Accuracy: 91.0
* 3k
1. Model: vit_base_patch16_224 | Accuracy: 91.36666666666666
2. Model: vit_tiny_patch16_224 | Accuracy: 91.1
3. Model: resnet18 | Accuracy: 91.73333333333333
4. Model: resnet50 | Accuracy: 91.36666666666666
5. Model: swin_base_patch4_window7_224 | Accuracy: 91.3